2.1 Does grid position matter more at Monaco than at other tracks?

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

df = pd.read_csv("/home/aris/projects/F1-Analysis/data/processed/results_joined.csv", low_memory=False)
df["circuit"].value_counts().head(10)

circuit
Autodromo Nazionale di Monza    1754
Circuit de Monaco               1508
Silverstone Circuit             1407
Circuit de Spa-Francorchamps    1259
Circuit Gilles Villeneuve       1026
Nürburgring                      953
Autódromo José Carlos Pace       926
Hungaroring                      911
Red Bull Ring                    910
Hockenheimring                   861
Name: count, dtype: int64

In [3]:
# Same filter as 1.4: finishers who started from a real grid spot
finishers = df[df["finished"] & (df["grid"] > 0)].copy()

# New column: True for rows at Monaco, False for every other track
finishers["is_monaco"] = finishers["circuit"] == "Circuit de Monaco"

# Same "link" score as 1.4 (1 = finish order matches grid order exactly)
def grid_finish_link(d):
    return d["grid"].corr(d["position"])

finishers.groupby("is_monaco").apply(grid_finish_link).round(2)

is_monaco
False    0.65
True     0.64
dtype: float64

In [4]:
rng = np.random.default_rng(42)  # random number maker; 42 = same "random" result every run

# The real gap: Monaco's link minus everyone else's
observed_gap = grid_finish_link(finishers[finishers["is_monaco"]]) - grid_finish_link(finishers[~finishers["is_monaco"]])

race_ids = finishers["raceId"].unique()                                # every race ID once
n_monaco = finishers.loc[finishers["is_monaco"], "raceId"].nunique()   # how many Monaco races (72)

fake_gaps = []
for i in range(1000):
    fake_races = rng.choice(race_ids, size=n_monaco, replace=False)  # pick 72 random races
    in_fake = finishers["raceId"].isin(fake_races)                   # True for rows in those races
    gap = grid_finish_link(finishers[in_fake]) - grid_finish_link(finishers[~in_fake])
    fake_gaps.append(gap)

fake_gaps = np.array(fake_gaps)
share = (np.abs(fake_gaps) >= abs(observed_gap)).mean()

print(f"Real Monaco gap: {observed_gap:.3f}")
print(f"Random groups of {n_monaco} races had a gap this big or bigger {share:.0%} of the time")

Real Monaco gap: -0.009
Random groups of 72 races had a gap this big or bigger 73% of the time


In [ ]:
# Front of the grid: does the pole sitter win more often at Monaco?
rng = np.random.default_rng(42)  # fresh random maker so this cell gives the same result every run

poles = df[df["pole"]].copy()
poles["is_monaco"] = poles["circuit"] == "Circuit de Monaco"

monaco_rate = poles.loc[poles["is_monaco"], "won"].mean()   # share of Monaco poles that became wins
other_rate = poles.loc[~poles["is_monaco"], "won"].mean()   # same for every other track
observed_pole_gap = monaco_rate - other_rate

pole_race_ids = poles["raceId"].unique()
n_monaco_poles = poles.loc[poles["is_monaco"], "raceId"].nunique()

fake_pole_gaps = []
for i in range(1000):
    fake_races = rng.choice(pole_race_ids, size=n_monaco_poles, replace=False)  # 72 random "fake Monacos"
    in_fake = poles["raceId"].isin(fake_races)
    fake_pole_gaps.append(poles.loc[in_fake, "won"].mean() - poles.loc[~in_fake, "won"].mean())

fake_pole_gaps = np.array(fake_pole_gaps)
pole_share = (np.abs(fake_pole_gaps) >= abs(observed_pole_gap)).mean()

print(f"Random groups of {n_monaco_poles} races had a gap this big or bigger {pole_share:.0%} of the time")
print(f"Pole sitter won at Monaco: {monaco_rate:.0%}, elsewhere: {other_rate:.0%}")

Pole sitter won at Monaco: 47%, elsewhere: 43%
Random groups of 72 races had a gap this big or bigger 59% of the time


Among cars that finished, the link between starting and finishing position at Monaco (0.64) is almost the same as at other tracks (0.65). A shuffle test found that random groups of 72 races show a gap this big or bigger 73% of the time, so the difference is easily explained by chance. The front of the grid tells the same story: pole sitters won 47% of the time at Monaco and 43% elsewhere, but random groups of 72 races show a gap that big or bigger 59% of the time. This data doesn't support the claim that grid position matters more at Monaco.